# 07 — Kaggle Dataset Ablation Training (KAGGLE GPU)
## WavSent-MTL · Tasks 3.13–3.16

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Same ablation (Configs A–F) as notebook 06, but on Kaggle dataset
- Kaggle dataset has n_features=9 (adds polarity_max)
- BEST_REPR must already be set in config.py from notebook 06
- Saves: `kaggle_ablation_AG.csv` + val/test prediction arrays

**PREREQUISITE:**
- Notebook 06 complete
- BEST_REPR set in config.py and pushed to GitHub
- Upload `data/processed/kaggle/` as Kaggle dataset: `wavsent-kaggle-processed`

In [1]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
import gc
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'BEST_REPR: {CONFIG["BEST_REPR"]}')

print()
print('─' * 45)
print('Training Configuration')
print('─' * 45)
print(f"Early stopping patience : {CONFIG['early_stopping_patience']}")
print(f"Early stopping monitor  : {CONFIG['early_stopping_monitor']}")
print(f"Max epochs              : {CONFIG['max_epochs']}")
print(f"LR reduce patience      : {CONFIG['lr_reduce_patience']}")
print(f"LR reduce monitor       : {CONFIG['lr_reduce_monitor']}")
print(f"LR reduce factor        : {CONFIG['lr_reduce_factor']}")
print(f"Loss type               : {CONFIG['loss_type']}")
print('─' * 45)

Cloning into '/kaggle/working/WavSent-MTL'...


PyTorch: 2.10.0+cu128
Device:  cuda
GPU:     Tesla T4
BEST_REPR: denoised_technicals

─────────────────────────────────────────────
Training Configuration
─────────────────────────────────────────────
Early stopping patience : 35
Early stopping monitor  : val_binary_accuracy
Max epochs              : 150
LR reduce patience      : 10
LR reduce monitor       : val_loss
LR reduce factor        : 0.5
Loss type               : uncertainty
─────────────────────────────────────────────


## Load Kaggle Processed Arrays

In [2]:
import numpy as np
import pandas as pd
import json

KAGGLE_INPUT = '/kaggle/input/datasets/consistency23/wavsent-kaggle-processed'

# Full data dict (Config C/BEST selected features — 9 features)
data_C = {
    'X_train':      np.load(f'{KAGGLE_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KAGGLE_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KAGGLE_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KAGGLE_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KAGGLE_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KAGGLE_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KAGGLE_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KAGGLE_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KAGGLE_INPUT}/y_reg_test.npy'),
}

with open(f'{KAGGLE_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)   # 9 features

with open(f'{KAGGLE_INPUT}/class_weights.json') as f:
    class_weights = json.load(f)

print(f'X_train shape:     {data_C["X_train"].shape}')
print(f'Selected features: {selected_features}')
print(f'Class weights:     {class_weights}')

X_train shape:     (1250, 5, 9)
Selected features: ['WILLIAMS_R', 'STOCH_K', 'MACD', 'RSI_14', 'CCI_20', 'EMA_9', 'Volume_d', 'polarity_mean', 'polarity_max']
Class weights:     None


## Build Config A and B Data Variants

In [3]:
from src.data.windows import create_windows, generate_targets, temporal_split
from src.data.preprocessor import apply_scaler, apply_reg_scaler

df_feat = pd.read_csv(f'{KAGGLE_INPUT}/featured_data.csv')
df_feat['Date'] = pd.to_datetime(df_feat['Date'])
df_feat = df_feat.sort_values('Date').reset_index(drop=True)

train_df, val_df, test_df = temporal_split(df_feat)
print(f'Split: Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')

def build_data_variant(feature_cols, train_df, val_df, test_df, w=5):
    tr_s, v_s, te_s, _ = apply_scaler(
        train_df[feature_cols].values,
        val_df[feature_cols].values,
        test_df[feature_cols].values,
        save_path='/tmp/tmp_scaler.pkl'
    )
    X_train = create_windows(tr_s, w)
    X_val   = create_windows(v_s, w)
    X_test  = create_windows(te_s, w)
    y_clf_tr, y_reg_tr = generate_targets(train_df['Close'].values, w)
    y_clf_v,  y_reg_v  = generate_targets(val_df['Close'].values, w)
    y_clf_te, y_reg_te = generate_targets(test_df['Close'].values, w)
    y_reg_tr, y_reg_v, y_reg_te, _ = apply_reg_scaler(
        y_reg_tr, y_reg_v, y_reg_te, save_path='/tmp/tmp_reg_scaler.pkl')
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_clf_train': y_clf_tr, 'y_clf_val': y_clf_v, 'y_clf_test': y_clf_te,
        'y_reg_train': y_reg_tr, 'y_reg_val': y_reg_v, 'y_reg_test': y_reg_te,
    }

# Config A: return + polarity_mean + polarity_max (Kaggle has both)
for split_df in [train_df, val_df, test_df]:
    split_df['return'] = split_df['Close'].pct_change().fillna(0)

CONFIG_A_FEATURES = ['return', 'polarity_mean', 'polarity_max']
data_A = build_data_variant(CONFIG_A_FEATURES, train_df, val_df, test_df)
print(f'Config A X_train: {data_A["X_train"].shape}  (3 features)')

# Config B: denoised OHLCV (5) + polarity_mean + polarity_max = 7 features
CONFIG_B_FEATURES = ['Open_d', 'High_d', 'Low_d', 'Close_d', 'Volume_d',
                     'polarity_mean', 'polarity_max']
data_B = build_data_variant(CONFIG_B_FEATURES, train_df, val_df, test_df)
print(f'Config B X_train: {data_B["X_train"].shape}  (7 features)')
print(f'Config C X_train: {data_C["X_train"].shape}  (9 features — already loaded)')

Split: Train=1255 | Val=269 | Test=270
Split: Train=1255 Val=269 Test=270
Config A X_train: (1250, 5, 3)  (3 features)
Config B X_train: (1250, 5, 7)  (7 features)
Config C X_train: (1250, 5, 9)  (9 features — already loaded)


## Helper: Run One Config

In [4]:
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/kaggle_ablation_partial.csv'
all_results = pd.DataFrame()


def run_config(config_name, model_name, data, class_weights=None):
    global all_results
    n_features = data['X_train'].shape[2]
    print(f'\n{"=" * 60}')
    print(f'Config {config_name} | {model_name.upper()} | n_features={n_features}')
    print(f'{"=" * 60}')

    results = train_multi_run(
        config_name=config_name,
        model_name=model_name,
        n_features=n_features,
        data=data,
        dataset='kaggle',
        class_weights=class_weights,
        device=device,
    )

    all_results = pd.concat([all_results, results], ignore_index=True)
    all_results.to_csv(RESULTS_PATH, index=False)

    mean_val  = results['val_accuracy'].mean()
    mean_test = results['accuracy'].mean()
    print(f'Config {config_name} | mean val_acc={mean_val:.4f} | mean test_acc={mean_test:.4f}')

    torch.cuda.empty_cache()
    gc.collect()
    return results


# Select BEST_REPR data
BEST_REPR = CONFIG['BEST_REPR']
data_BEST = data_B if BEST_REPR == 'denoised_ohlcv' else data_C
print(f'BEST_REPR: {BEST_REPR} | n_features={data_BEST["X_train"].shape[2]}')

BEST_REPR: denoised_technicals | n_features=9


## Run All Configs A–F

In [5]:
results_A = run_config('A', 'tkan', data_A, class_weights)


Config A | TKAN | n_features=3
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 1/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 2/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 3/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 4/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 5/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 6/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 7/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config A | tkan | Run 8/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.49

In [6]:
results_B = run_config('B', 'tkan', data_B, class_weights)


Config B | TKAN | n_features=7
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 1/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 2/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 3/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 4/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 5/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 6/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 7/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config B | tkan | Run 8/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 36 / 150 | Best val_acc: 0.49

In [7]:
results_C = run_config('C', 'tkan', data_C, class_weights)


Config C | TKAN | n_features=9
  Stopped at epoch 132 / 150 | Best val_acc: 0.6477
Config C | tkan | Run 1/30 | Val acc: 0.6477 | Test acc: 0.6981
  Stopped at epoch 121 / 150 | Best val_acc: 0.6629
Config C | tkan | Run 2/30 | Val acc: 0.6629 | Test acc: 0.6792
  Stopped at epoch 150 / 150 | Best val_acc: 0.6780
Config C | tkan | Run 3/30 | Val acc: 0.6780 | Test acc: 0.6868
  Stopped at epoch 123 / 150 | Best val_acc: 0.6591
Config C | tkan | Run 4/30 | Val acc: 0.6591 | Test acc: 0.6830
  Stopped at epoch 125 / 150 | Best val_acc: 0.6402
Config C | tkan | Run 5/30 | Val acc: 0.6402 | Test acc: 0.6868
  Stopped at epoch 150 / 150 | Best val_acc: 0.6856
Config C | tkan | Run 6/30 | Val acc: 0.6856 | Test acc: 0.6981
  Stopped at epoch 82 / 150 | Best val_acc: 0.6439
Config C | tkan | Run 7/30 | Val acc: 0.6439 | Test acc: 0.6566
  Stopped at epoch 119 / 150 | Best val_acc: 0.6250
Config C | tkan | Run 8/30 | Val acc: 0.6250 | Test acc: 0.6943
  Stopped at epoch 127 / 150 | Best val_a

In [8]:
results_D = run_config('D', 'lstm', data_BEST, class_weights)


Config D | LSTM | n_features=9
  Stopped at epoch 106 / 150 | Best val_acc: 0.7235
Config D | lstm | Run 1/30 | Val acc: 0.7235 | Test acc: 0.6566
  Stopped at epoch 91 / 150 | Best val_acc: 0.7500
Config D | lstm | Run 2/30 | Val acc: 0.7500 | Test acc: 0.6943
  Stopped at epoch 97 / 150 | Best val_acc: 0.7311
Config D | lstm | Run 3/30 | Val acc: 0.7311 | Test acc: 0.6792
  Stopped at epoch 74 / 150 | Best val_acc: 0.7121
Config D | lstm | Run 4/30 | Val acc: 0.7121 | Test acc: 0.6792
  Stopped at epoch 94 / 150 | Best val_acc: 0.7311
Config D | lstm | Run 5/30 | Val acc: 0.7311 | Test acc: 0.6755
  Stopped at epoch 36 / 150 | Best val_acc: 0.4924
Config D | lstm | Run 6/30 | Val acc: 0.4924 | Test acc: 0.5962
  Stopped at epoch 114 / 150 | Best val_acc: 0.7311
Config D | lstm | Run 7/30 | Val acc: 0.7311 | Test acc: 0.6792
  Stopped at epoch 95 / 150 | Best val_acc: 0.7159
Config D | lstm | Run 8/30 | Val acc: 0.7159 | Test acc: 0.6717
  Stopped at epoch 90 / 150 | Best val_acc: 0.

In [9]:
results_E = run_config('E', 'gru', data_BEST, class_weights)


Config E | GRU | n_features=9
  Stopped at epoch 66 / 150 | Best val_acc: 0.6932
Config E | gru | Run 1/30 | Val acc: 0.6932 | Test acc: 0.6679
  Stopped at epoch 77 / 150 | Best val_acc: 0.7083
Config E | gru | Run 2/30 | Val acc: 0.7083 | Test acc: 0.6792
  Stopped at epoch 80 / 150 | Best val_acc: 0.6932
Config E | gru | Run 3/30 | Val acc: 0.6932 | Test acc: 0.6642
  Stopped at epoch 91 / 150 | Best val_acc: 0.6894
Config E | gru | Run 4/30 | Val acc: 0.6894 | Test acc: 0.6302
  Stopped at epoch 60 / 150 | Best val_acc: 0.6818
Config E | gru | Run 5/30 | Val acc: 0.6818 | Test acc: 0.6792
  Stopped at epoch 91 / 150 | Best val_acc: 0.6932
Config E | gru | Run 6/30 | Val acc: 0.6932 | Test acc: 0.6491
  Stopped at epoch 72 / 150 | Best val_acc: 0.6932
Config E | gru | Run 7/30 | Val acc: 0.6932 | Test acc: 0.6528
  Stopped at epoch 86 / 150 | Best val_acc: 0.7121
Config E | gru | Run 8/30 | Val acc: 0.7121 | Test acc: 0.6453
  Stopped at epoch 89 / 150 | Best val_acc: 0.7008
Config

In [10]:
results_F = run_config('F', 'tcn', data_BEST, class_weights)


Config F | TCN | n_features=9
  Stopped at epoch 53 / 150 | Best val_acc: 0.6856
Config F | tcn | Run 1/30 | Val acc: 0.6856 | Test acc: 0.6302
  Stopped at epoch 76 / 150 | Best val_acc: 0.6780
Config F | tcn | Run 2/30 | Val acc: 0.6780 | Test acc: 0.6528
  Stopped at epoch 63 / 150 | Best val_acc: 0.6818
Config F | tcn | Run 3/30 | Val acc: 0.6818 | Test acc: 0.6340
  Stopped at epoch 94 / 150 | Best val_acc: 0.7008
Config F | tcn | Run 4/30 | Val acc: 0.7008 | Test acc: 0.6302
  Stopped at epoch 68 / 150 | Best val_acc: 0.6856
Config F | tcn | Run 5/30 | Val acc: 0.6856 | Test acc: 0.6491
  Stopped at epoch 55 / 150 | Best val_acc: 0.6894
Config F | tcn | Run 6/30 | Val acc: 0.6894 | Test acc: 0.6453
  Stopped at epoch 50 / 150 | Best val_acc: 0.6705
Config F | tcn | Run 7/30 | Val acc: 0.6705 | Test acc: 0.6415
  Stopped at epoch 60 / 150 | Best val_acc: 0.6818
Config F | tcn | Run 8/30 | Val acc: 0.6818 | Test acc: 0.6792
  Stopped at epoch 69 / 150 | Best val_acc: 0.6856
Config

## Summary

In [11]:
final_path = '/kaggle/working/kaggle_ablation_AG.csv'
all_results.to_csv(final_path, index=False)

print('=' * 60)
print('Notebook 07 -- Kaggle Training: COMPLETE')
print('=' * 60)

summary = all_results.groupby('config').agg(
    mean_val_acc=('val_accuracy', 'mean'),
    mean_test_acc=('accuracy', 'mean'),
    std_test_acc=('accuracy', 'std'),
    mean_auc=('auc', 'mean'),
).round(4)
print(summary.to_string())

print(f'\nDownload these files from /kaggle/working/:')
print('  kaggle_ablation_AG.csv')
print()
print('  kaggle_tkan_best.pt          ← best TKAN weights (for SHAP)')
print('  kaggle_lstm_best.pt          ← best LSTM weights (for SHAP)')
print('  kaggle_gru_best.pt           ← best GRU  weights (for SHAP)')
print('  kaggle_tcn_best.pt           ← best TCN  weights (for SHAP)')
print()
print('Val predictions saved at:')
print('  ablation/results/kaggle/val_predictions/*.npy')
print()
print('After downloading .pt files, place them at:')
print('  results/saved_models/kaggle/{model}_best.pt')
print()
print('Next: run notebook 08_ensemble')

Notebook 07 -- Kaggle Training: COMPLETE
        mean_val_acc  mean_test_acc  std_test_acc  mean_auc
config                                                     
A             0.4924         0.5962        0.0000    0.5065
B             0.4924         0.5962        0.0000    0.5468
C             0.6511         0.6766        0.0208    0.7134
D             0.7218         0.6702        0.0202    0.7155
E             0.6991         0.6630        0.0151    0.7168
F             0.6871         0.6452        0.0214    0.7063

Download these files from /kaggle/working/:
  kaggle_ablation_AG.csv

  kaggle_tkan_best.pt          ← best TKAN weights (for SHAP)
  kaggle_lstm_best.pt          ← best LSTM weights (for SHAP)
  kaggle_gru_best.pt           ← best GRU  weights (for SHAP)
  kaggle_tcn_best.pt           ← best TCN  weights (for SHAP)

Val predictions saved at:
  ablation/results/kaggle/val_predictions/*.npy

After downloading .pt files, place them at:
  results/saved_models/kaggle/{model}_be